In [0]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.window import Window

import pyspark.sql.functions as f
import pyspark.sql.types as t

In [0]:
dbutils.widgets.dropdown("position_filter", "WR", ["WR", "RB", "TE", "QB"])

In [0]:
# load parquet files into dataframes
df_25 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_25/stats_player_reg_2025.parquet")
df_24 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_24/stats_player_reg_2024.parquet")
df_23 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_23/stats_player_reg_2023.parquet")
df_22 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_22/stats_player_reg_2022.parquet")
df_25.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'def_tackles_solo',
 'def_tackles_with_assist',
 '

In [0]:
# create union for all dataframes and add hppr files
total_df = df_25.union(df_24).union(df_23).union(df_22)
total_df = total_df.withColumn("hppr_points", (f.col("fantasy_points_ppr")+f.col("fantasy_points"))/2
                               ).withColumn("hppr_ppg", ((f.col("fantasy_points_ppr")+f.col("fantasy_points"))/2) / f.col("games"))
total_df = total_df.orderBy("hppr_ppg", ascending=False)

In [0]:
# create temporary view for SQL
total_df.createOrReplaceTempView("total_sql")

In [0]:
%sql
-- group WRs and find all time PPG

with players_grouped as (
    select
        player_display_name,
        position
    from
        total_sql
    group by player_display_name, position
)

select 
    pg.player_display_name, 
    pg.position,
    MAX(recent_team),
    ROUND(SUM(hppr_points) / SUM(games), 2) as avg_season_ppg,
    ROUND(SUM(receiving_yards) / SUM(games), 2) as avg_rec_ypg,
    ROUND(SUM(receiving_tds) / SUM(games), 2) as avg_rec_tds_pg,
    ROUND(SUM(rushing_yards) / SUM(games), 2) as avg_rush_ypg,
    ROUND(SUM(rushing_tds) / SUM(games), 2) as avg_rush_tds_pg,
    ROUND(SUM(passing_yards) / SUM(games), 2) as avg_pass_ypg,
    ROUND(SUM(passing_tds) / SUM(games), 2) as avg_pass_tds_pg
FROM players_grouped pg
JOIN total_sql t ON t.player_display_name = pg.player_display_name AND t.position = pg.position
WHERE t.games > 7 AND pg.position == :position_filter
GROUP BY ALL
ORDER BY avg_season_ppg DESC

player_display_name,position,MAX(recent_team),avg_season_ppg,avg_rec_ypg,avg_rec_tds_pg,avg_rush_ypg,avg_rush_tds_pg,avg_pass_ypg,avg_pass_tds_pg
Puka Nacua,WR,LA,16.45,95.25,0.43,5.45,0.05,0.0,0.0
Ja'Marr Chase,WR,CIN,16.43,88.23,0.67,0.79,0.0,-0.11,0.0
Tyreek Hill,WR,MIA,15.53,89.36,0.52,2.0,0.02,0.0,0.0
CeeDee Lamb,WR,DAL,15.49,86.76,0.48,3.74,0.03,0.0,0.0
Amon-Ra St. Brown,WR,DET,15.31,80.91,0.59,2.03,0.0,0.11,0.02
Justin Jefferson,WR,MIN,14.73,89.57,0.41,0.36,0.02,0.92,0.0
Malik Nabers,WR,NYG,14.61,80.27,0.47,0.13,0.0,0.0,0.0
Davante Adams,WR,NYJ,14.37,72.77,0.71,-0.02,0.0,0.0,0.0
A.J. Brown,WR,PHI,13.82,81.19,0.52,0.0,0.0,0.0,0.0
Mike Evans,WR,TB,13.02,69.46,0.61,0.0,0.0,0.0,0.0
